In [1]:
# Cell 1: Read the silver_sales Delta table
df_silver = spark.read.format('delta').load('Tables/dbo/silver_sales')

print(f'Row count: {df_silver.count()}')
print(f'Columns: {df_silver.columns}')
display(df_silver.limit(5))

StatementMeta(, b20b9ff2-2097-4860-8077-f1f52d668edf, 3, Finished, Available, Finished, False)

Row count: 1268
Columns: ['order_id', 'order_date', 'customer_name', 'region', 'product_category', 'revenue', 'quantity', 'status', 'revenue_usd']


SynapseWidget(Synapse.DataFrame, cdc601ca-ac76-4d88-b4ff-c3790c0d0d78)

In [2]:
# Cell 2: Extract year and month from order_date
from pyspark.sql.functions import col, to_date, date_format, sum, count, round, avg

# Convert order_date string to proper DateType first
df_dated = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

# Add year_month column for grouping
df_dated = df_dated.withColumn(
    'year_month',
    date_format(col('order_date'), 'yyyy-MM')
)

display(df_dated.select('order_id', 'order_date', 'year_month').limit(5))


StatementMeta(, b20b9ff2-2097-4860-8077-f1f52d668edf, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05925787-c492-4a7f-9b51-bf72c11828ec)

In [3]:
# Cell 3: Aggregate to Gold layer
df_gold = df_dated.groupBy('region', 'year_month') \
    .agg(
        round(sum('revenue'), 2).alias('total_revenue'),
        count('order_id').alias('total_orders'),
        round(avg('revenue'), 2).alias('avg_order_value')
    ) \
    .orderBy('year_month', 'region')

print(f'Gold table row count: {df_gold.count()}')
display(df_gold)


StatementMeta(, b20b9ff2-2097-4860-8077-f1f52d668edf, 5, Finished, Available, Finished, False)

Gold table row count: 96


SynapseWidget(Synapse.DataFrame, 379972c2-6c91-443e-b733-0dea4dd79354)

In [4]:
# Cell 4: Write the Gold layer to a Delta table
df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('gold_sales_summary')


StatementMeta(, b20b9ff2-2097-4860-8077-f1f52d668edf, 6, Finished, Available, Finished, False)